In [1]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV10FM
from tscglue import data_loader
import polars as pl
from sklearn.metrics import log_loss

In [ ]:
dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
#dataset = 'Trace'
#dataset = 'SwedishLeaf'
#dataset = 'Meat'
# dataset='ACSF1'
# dataset='ElectricDevices'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 19)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(8926, 1, 96) (8926,) (7711, 1, 96) (7711,)


In [13]:
seed = 2683

In [14]:
from tscglue.models import LokyStackerV10RSTSFRandom, TSCGlueClassifier

m = TSCGlueClassifier(random_state=seed, n_jobs=16, verbose=10)

In [15]:
m.fit(X_train, y_train)

[0.00s] Starting fit, run_dir=tscglue_runs/cce7031af6834ced, n_jobs=16
[0.01s] Saved X and y to disk in 0.01s (dtype=float64)
[15.69s] Fit+transformed multirocket_s_1675711793 features (8926, 49728) (3386.48 MB) dtype=float64 in 15.6444s
[subprocess] fit_transform hydra_s_2109509380: 6.1574s
[26.99s] Fit+transformed hydra_s_2109509380 features (8926, 4096) (278.94 MB) dtype=float64 in 11.2953s
[subprocess] fit_transform quant: 15.3944s
[47.29s] Fit+transformed quant features (8926, 994) (67.69 MB) dtype=float64 in 20.2974s
[132.31s] Fit+transformed rdst_s_1973074453 features (8926, 30000) (2043.00 MB) dtype=float64 in 85.0260s


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:836: RuntimeWarning: divide by zero encountered in scalar divide
  pacf[i + 1] = 2 / d[i + 1] * v[i + 1 :].dot(u[i:-1])
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:836: RuntimeWarning: invalid value encountered in scalar multiply
  pacf[i + 1] = 2 / d[i + 1] * v[i + 1 :].dot(u[i:-1])


[214.48s] Fit+transformed rstsf-random_s_447740098 features (8926, 11172) (760.81 MB) dtype=float64 in 82.1657s
[subprocess] fit_transform mantis: 245.3060s
[468.25s] Fit+transformed mantis features (8926, 512) (34.87 MB) dtype=float64 in 253.7721s
[subprocess] fit_transform chronos2: 388.7166s
[865.56s] Fit+transformed chronos2 features (8926, 1536) (104.60 MB) dtype=float64 in 397.3044s
[865.60s] Starting training with 16 workers for 50 models
[926.05s] Trained quant-etc_s_1672401463 in 60.4301s for f-0 (56.02 MB)
[927.01s] Trained quant-etc_s_1672401463 in 61.3741s for f-1 (56.42 MB)
[931.40s] Trained quant-etc_s_1672401463 in 65.7647s for f-4 (56.31 MB)
[932.01s] Trained quant-etc_s_1672401463 in 66.3704s for f-5 (56.53 MB)
[932.32s] Trained quant-etc_s_1672401463 in 66.6914s for f-2 (56.39 MB)
[940.75s] Trained quant-etc_s_1672401463 in 75.1158s for f-3 (56.46 MB)
[982.22s] Trained quant-etc_s_1672401463 in 56.1487s for f-6 (56.11 MB)
[991.53s] Trained quant-etc_s_1672401463 in 64

,random_state,2683
,k_folds,10
,n_jobs,16
,verbose,10
,n_repetitions,1


In [16]:
pl.DataFrame(m.summary(return_transforms=True)).head(8)

model,level,oof_accuracy,train_time
str,i64,f64,f64
"""multirocket_s_1675711793""",null,null,15.644357
"""hydra_s_2109509380""",null,null,11.295338
"""quant""",null,null,20.29741
"""rdst_s_1973074453""",null,null,85.026002
"""rstsf-random_s_447740098""",null,null,82.165732
"""mantis""",null,null,253.772054
"""chronos2""",null,null,397.304351
"""quant-etc_s_1672401463""",0,0.884719,646.197955


In [8]:
m.classes_

array(['1', '2', '3', '4', '5'], dtype='<U1')

In [9]:
sdfsdf=Sdfgsg

NameError: name 'Sdfgsg' is not defined

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
proba = m.predict_proba(X_test)
classes = list(m.classes_)

print(f"Log-loss: {log_loss(y_test, proba, labels=classes):.4f}")
# print(f"AUC (OvR): {roc_auc_score(y_test, proba, multi_class='ovr', labels=classes):.4f}")

In [ ]:
pl.DataFrame(m.neki)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Build probability matrix from level-0 neki predictions
df = pl.DataFrame(m.neki).filter(pl.col("level") == 0)
prob_matrix = df.pivot(on="class", index=["index", "model"], values="probability", aggregate_function='mean')
class_cols = [c for c in prob_matrix.columns if c not in ("index", "model")]
wide = prob_matrix.pivot(on="model", index="index", values=class_cols)
wide = wide.sort("index")
X_stack = wide.drop("index").to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y_train)

stackers = {
    "RidgeCV": RidgeClassifierCV(alphas=np.logspace(-3, 3, 20)),
    "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1),
    "HGBT": HistGradientBoostingClassifier(max_iter=200),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500),
}

N_REPEATS = 20
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=N_REPEATS, random_state=42)
results = {name: [] for name in stackers}

for fold_i, (tr_idx, val_idx) in enumerate(rskf.split(X_stack, y_enc)):
    rep = fold_i // 10
    for name, clf in stackers.items():
        clf_copy = clf.__class__(**clf.get_params())
        clf_copy.fit(X_stack[tr_idx], y_enc[tr_idx])
        acc = accuracy_score(y_enc[val_idx], clf_copy.predict(X_stack[val_idx]))
        results[name].append({"rep": rep, "fold": fold_i % 10, "acc": acc})

# Build test probability matrix
test_preds = m.predict_proba_per_model(X_test)
train_cols = wide.drop("index").columns

test_records = []
for model_name, proba_arr in test_preds.items():
    if model_name in m.stacking_models:
        continue
    classes_ = list(m.classes_)
    for i in range(proba_arr.shape[0]):
        for j, cls in enumerate(classes_):
            test_records.append({"index": i, "model": model_name, "level": 0, "class": str(cls), "probability": proba_arr[i, j]})

df_test = pl.DataFrame(test_records)
prob_matrix_test = df_test.pivot(on="class", index=["index", "model"], values="probability")
class_cols_test = [c for c in prob_matrix_test.columns if c not in ("index", "model")]
wide_test = prob_matrix_test.pivot(on="model", index="index", values=class_cols_test)
wide_test = wide_test.sort("index")
X_stack_test = wide_test.select(train_cols).to_numpy()
y_test_enc = le.transform(y_test)

# Test set accuracy
test_accs = {}
for name, clf in stackers.items():
    clf_final = clf.__class__(**clf.get_params())
    clf_final.fit(X_stack, y_enc)
    test_accs[name] = accuracy_score(y_test_enc, clf_final.predict(X_stack_test))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names = list(stackers.keys())
x_positions = np.arange(len(names))

for i, name in enumerate(names):
    rdf = pl.DataFrame(results[name])
    per_rep = rdf.group_by("rep").agg(pl.col("acc").mean()).sort("rep")["acc"].to_numpy()
    # scatter all per-rep OOF accuracies
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(per_rep))
    ax.scatter(np.full_like(per_rep, i) + jitter, per_rep, alpha=0.5, s=40, zorder=3, label=f"{name} OOF reps" if i == 0 else None, color=f"C{i}")
    # mean OOF line
    ax.hlines(per_rep.mean(), i - 0.25, i + 0.25, colors=f"C{i}", linewidths=2, zorder=4)
    # test accuracy marker
    ax.scatter(i, test_accs[name], marker="*", s=200, color=f"C{i}", edgecolors="black", linewidths=0.8, zorder=5)
    print(f"{name:8s}  OOF mean={per_rep.mean():.4f} std={per_rep.std():.4f}  test={test_accs[name]:.4f}")

ax.set_xticks(x_positions)
ax.set_xticklabels(names)
ax.set_ylabel("Accuracy")
ax.set_title(f"Stacker comparison: OOF ({N_REPEATS}x10-fold) vs Test")
# custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="OOF per-rep mean"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=15, label="Test accuracy"),
    Line2D([0], [0], color="gray", linewidth=2, label="OOF overall mean"),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
